# Rétention Client – Machine Learning
**Dataset :** `customer_churn.csv` (10 000 clients)  
**Objectif principal :** Prédire `churn` (classification binaire : 0 = No, 1 = Yes)  
**Modèles testés :** Logistic Regression, Random Forest, XGBoost, MLP (Deep Learning)  
**Métriques prioritaires :** Recall, F1-score, ROC-AUC, PR-AUC (classes déséquilibrées ~10 % churn)

## 1. Importation des librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Librairies EDA chargées OK')

## 2. Chargement des données

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')
print(f'Shape : {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Analyse exploratoire (EDA)

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Valeurs manquantes :')
print(missing)

if not missing.empty:
    missing.plot(kind='bar', figsize=(8, 4), title='Valeurs manquantes par colonne', color='tomato')
    plt.ylabel('Nombre de valeurs manquantes')
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution de la variable cible (déséquilibre de classes)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['churn'].value_counts()
counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Distribution churn')
axes[0].set_xticklabels(['No Churn (0)', 'Churn (1)'], rotation=0)
for i, v in enumerate(counts):
    axes[0].text(i, v + 50, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

df['contract_type'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Répartition par type de contrat')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f"Ratio déséquilibre : {counts[0]/counts[1]:.1f}:1 (No Churn : Churn)")

In [ ]:
# Distribution des variables numériques clés
num_cols = ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days',
            'avg_session_time', 'monthly_fee', 'total_revenue',
            'payment_failures', 'support_tickets', 'nps_score']

df[num_cols].hist(figsize=(16, 8), bins=40, edgecolor='white', color='steelblue')
plt.suptitle('Distribution des variables numériques', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation des variables numériques
corr_cols = ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days',
             'avg_session_time', 'features_used', 'usage_growth_rate',
             'last_login_days_ago', 'monthly_fee', 'total_revenue',
             'payment_failures', 'support_tickets', 'csat_score',
             'nps_score', 'churn']

plt.figure(figsize=(12, 9))
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5,
            annot_kws={'size': 8})
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots : variables numériques vs churn
key_cols = ['tenure_months', 'monthly_logins', 'payment_failures',
            'nps_score', 'csat_score', 'support_tickets',
            'usage_growth_rate', 'last_login_days_ago']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(key_cols):
    df.boxplot(column=col, by='churn', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Churn')

plt.suptitle('Variables numériques vs Churn', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Taux de churn par variables catégorielles
cat_cols = ['contract_type', 'customer_segment', 'payment_method', 
            'discount_applied', 'price_increase_last_3m']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
    ax.set_title(f'Taux de churn par {col}')
    ax.set_ylabel('Taux de churn')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Taux de churn par variable catégorielle', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Pipeline complet (preprocessing → entraînement → évaluation)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing import load_data, engineer_features, get_train_test_split, FEATURES, TARGET
from src.train import build_pipelines, cross_validate_models, train_and_save
from src.evaluate import (
    evaluate_models, plot_confusion_matrices, plot_roc_curves,
    plot_pr_curves, plot_feature_importance, plot_threshold_analysis,
)

### 4.1 Préparation des données

In [ ]:
df_clean = load_data()
df_clean = engineer_features(df_clean)
X_train, X_test, y_train, y_test = get_train_test_split(df_clean)

print(f'Train : {X_train.shape[0]} lignes  |  Test : {X_test.shape[0]} lignes')
print(f'Taux de churn (train) : {y_train.mean():.2%}')
print(f'Features utilisées : {len(FEATURES)}')
print(FEATURES)

### 4.2 Cross-validation (StratifiedKFold 5 folds)

On utilise **StratifiedKFold** pour préserver les proportions de la classe minoritaire (~10 % churn) dans chaque fold.

In [ ]:
models = build_pipelines()
cv_results = cross_validate_models(models, X_train, y_train)

cv_df = pd.DataFrame(cv_results)
print(cv_df.agg(['mean', 'std']).round(4))

cv_df.plot(kind='box', figsize=(8, 5))
plt.title('ROC-AUC (Cross-Validation StratifiedKFold 5 folds)')
plt.ylabel('ROC-AUC')
plt.tight_layout()
plt.show()

### 4.3 Entraînement final + sauvegarde

In [ ]:
models_dir = os.path.join('..', 'models')
trained_models = train_and_save(models, X_train, y_train, model_dir=models_dir)

### 4.4 Évaluation sur le jeu de test

Métriques utilisées : **Recall** et **F1-score** prioritaires (faux négatifs = churners non détectés = coût élevé), ROC-AUC et PR-AUC.

In [ ]:
results_df = evaluate_models(trained_models, X_test, y_test)
results_df

In [ ]:
fig_cm = plot_confusion_matrices(trained_models, X_test, y_test)
plt.show()

In [ ]:
fig_roc = plot_roc_curves(trained_models, X_test, y_test)
plt.show()

In [ ]:
# Courbes Precision-Recall : plus informatives que ROC avec classes déséquilibrées
fig_pr = plot_pr_curves(trained_models, X_test, y_test)
plt.show()

### 4.5 Analyse du seuil de décision

Par défaut, le seuil est 0.5. Dans un contexte déséquilibré, abaisser le seuil permet d'augmenter le Recall (détecter plus de churners) au prix d'une Precision plus faible.

In [ ]:
fig_thr = plot_threshold_analysis(trained_models, X_test, y_test)
plt.show()

### 4.6 Importance des features (Random Forest & XGBoost)

In [ ]:
fig_fi, feat_imp_df = plot_feature_importance(trained_models, FEATURES)
if fig_fi is not None:
    plt.show()
feat_imp_df

## 5. Conclusion

### Résumé des modèles testés

| Modèle | Caractéristiques | Forces | Limites |
|--------|-----------------|--------|--------|
| **Logistic Regression** | Linéaire, interprétable | Robuste, rapide, baseline solide | Ne capture pas les non-linéarités |
| **Random Forest** | Ensemble d'arbres | Bonne généralisation, feature importance | Moins interprétable |
| **XGBoost** | Gradient Boosting | Performances élevées | Requiert tuning |
| **MLP** | Réseau de neurones | Capture interactions complexes | Instable avec faible déséquilibre, coût computationnel |

### Gestion du déséquilibre de classes (~10 % churn)
- `class_weight='balanced'` appliqué aux modèles LR et RF
- `scale_pos_weight=8` pour XGBoost
- Validation croisée stratifiée (StratifiedKFold)
- Métriques adaptées : Recall, F1-score, PR-AUC

### Modèle recommandé
**Random Forest** – meilleur compromis ROC-AUC / stabilité / interprétabilité dans ce contexte de rétention client.

### Pistes d'amélioration
- Appliquer SMOTE ou Random Over-Sampling pour rééquilibrer les classes
- Optimisation des hyperparamètres (RandomizedSearchCV)
- Ajouter SHAP pour l'explicabilité locale (pourquoi ce client est à risque ?)
- Ajuster le seuil de décision pour maximiser le Recall
- Tester Gradient Boosting / LightGBM